[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sampriti-sr/MLP_Course/blob/kshitij-variant/docs/toolkit/sklearn-preprocessing.ipynb)

# scikit-learn: Preprocessing

<div class="admonition info">
<p class="admonition-title">What you will be able to do after this page</p>
<ul>
<li>Know which model families need scaling and which do not.</li>
<li>Choose between scalers by how they treat outliers, and check what a transform did.</li>
<li>Encode categories properly, including unseen and rare ones.</li>
<li>Impute missing values and know when an indicator helps.</li>
</ul>
</div>

**Prerequisites:** <a href="../sklearn-api/">scikit-learn: The Estimator API</a>. **Dataset:** Credit Card Default (Taiwan, 2005). **Time:** about 45 minutes.

In [1]:
try:
    import mlp_data
except ImportError:  # e.g. a fresh Colab runtime
    %pip install -q "mlp-course @ git+https://github.com/sampriti-sr/MLP_Course@kshitij-variant"
    import mlp_data

import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlp_data import load_credit, style

style.apply()
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 14)
pd.set_option("display.float_format", "{:,.3f}".format)

## The situation

Raw columns are rarely in the form a model wants. Amounts span wildly different ranges, categories arrive as numbers that are not really numbers, and values go missing. **Preprocessing** reshapes the inputs. The skill is knowing which models care about which fix, so you do not waste effort (or, worse, break something) where it does not matter.

We use the credit-default data again. Set up the split once so every comparison below is fair:

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

credit = load_credit()
y = credit["default payment next month"]
X = credit.drop(columns=["ID", "default payment next month"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

def auc(model):
    """Fit on the training rows, return the AUC on the test rows."""
    return roc_auc_score(y_test, model.fit(X_train, y_train).predict_proba(X_test)[:, 1])

X[["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1"]].describe().loc[["min", "50%", "max"]].round(0)

,LIMIT_BAL,AGE,BILL_AMT1,PAY_AMT1
min,"10,000.000",21.000,"-165,580.000",0.000
50%,"140,000.000",34.000,"22,382.000","2,100.000"
max,"1,000,000.000",79.000,"964,511.000","873,552.000"


Look at the ranges: credit limits run into the hundreds of thousands, ages sit between 21 and 79, and bills and payments vary by orders of magnitude (a bill can even be negative when the customer overpaid). A model that compares or adds these numbers will treat a difference of 1,000 in a bill the same as a difference of 1,000 years of age.

## 1. Scaling: which models care?

Fit several families on the raw features and then on standardised features, and see whose score changes:

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.exceptions import ConvergenceWarning

def auc_and_converged(model):
    """AUC, plus whether the fit finished without a ConvergenceWarning (the warning text can include file paths, so we capture it)."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        score = auc(model)
    return score, not any(issubclass(w.category, ConvergenceWarning) for w in caught)

families = {"logistic regression": lambda: LogisticRegression(max_iter=3000), "k-nearest neighbours": lambda: KNeighborsClassifier(25),
            "decision tree (depth 4)": lambda: DecisionTreeClassifier(max_depth=4, random_state=0), "gradient boosting": lambda: HistGradientBoostingClassifier(random_state=0)}
rows = {}
for name, make in families.items():
    raw_auc, raw_ok = auc_and_converged(make())
    scaled_auc, scaled_ok = auc_and_converged(make_pipeline(StandardScaler(), make()))
    rows[name] = {"unscaled AUC": raw_auc, "standardised AUC": scaled_auc, "change": scaled_auc - raw_auc, "unscaled fit converged": raw_ok}
scaling_effect = pd.DataFrame(rows).T
scaling_effect

,unscaled AUC,standardised AUC,change,unscaled fit converged
logistic regression,0.711,0.711,-0.000,False
k-nearest neighbours,0.652,0.738,0.086,True
decision tree (depth 4),0.734,0.734,0.000,True
gradient boosting,0.774,0.774,0.000,True


**One model cares enormously, and three do not.** k-nearest neighbours jumps from about 0.65 to 0.74, because it works from *distances*, and with raw units the distance is dominated by whichever columns have the biggest numbers (credit limits and bills), drowning out the repayment status that actually predicts default. Trees only compare a feature with a threshold, so monotone rescaling cannot change their splits. Gradient boosting is built from trees, so it is also indifferent. Logistic regression's score is unchanged to three decimals, **but look at the last column**: the unscaled fit did not converge (scikit-learn raised a `ConvergenceWarning` after 3,000 iterations). The optimiser struggles when features live on wildly different scales, and here it happened to land on a solution about as good as the converged one. Do not rely on luck: scaling makes the optimisation well-behaved and makes coefficients comparable.

| Model family | Needs scaling? | Why |
|---|---|---|
| k-NN, SVM, k-means, PCA, neural nets | **yes** | distances, dot products or gradients depend on units |
| Linear / logistic regression | helpful | faster convergence, comparable coefficients, essential with regularisation |
| Trees, forests, gradient boosting | no | only the *order* of values matters |

## 2. Which scaler?

Three common ones differ in what they do to **outliers**. Take the bill amount, which has some very large values:

In [4]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

bill = X[["BILL_AMT1"]]
rows = {}
for name, scaler in [("StandardScaler (mean 0, sd 1)", StandardScaler()), ("MinMaxScaler (0 to 1)", MinMaxScaler()), ("RobustScaler (median 0, IQR 1)", RobustScaler())]:
    z = scaler.fit_transform(bill).ravel()
    rows[name] = {"minimum": z.min(), "median": np.median(z), "maximum": z.max(), "share beyond 3 units from the median": (np.abs(z - np.median(z)) > 3).mean()}
pd.DataFrame(rows).T

,minimum,median,maximum,share beyond 3 units from the median
"StandardScaler (mean 0, sd 1)",-2.944,-0.392,12.403,0.031
MinMaxScaler (0 to 1),0.000,0.166,1.000,0.000
"RobustScaler (median 0, IQR 1)",-2.959,0.000,14.829,0.043


- **StandardScaler** subtracts the mean and divides by the standard deviation. The default choice, but the mean and standard deviation are themselves pulled around by extreme values.
- **MinMaxScaler** squeezes everything into 0 to 1. A single huge outlier squashes all the ordinary values into a sliver near 0 (here the median lands at about 0.17).
- **RobustScaler** uses the median and the interquartile range, which barely move when outliers appear. The outliers remain visible (the maximum is far from the median) but they no longer distort how the ordinary values are scaled.

On this data the choice barely matters for k-NN (all three give AUC within about 0.005 of each other), but with heavier outliers the robust scaler is the safe option.

## 3. Skew: a transform is not automatically a cure

A heavily skewed column is often tamed with a log. Bill amounts include zeros and negatives, so a plain log fails; two standard answers are a **signed log** and the **Yeo–Johnson power transform**. *Check what they did*:

In [5]:
from sklearn.preprocessing import PowerTransformer

b = X["BILL_AMT1"]
signed_log = np.sign(b) * np.log1p(np.abs(b))
yeo_johnson = pd.Series(PowerTransformer().fit_transform(b.to_frame()).ravel(), index=b.index)
print(f"zero bills: {(b == 0).mean():.1%}; negative bills: {(b < 0).mean():.1%}")
pd.Series({"raw": b.skew(), "signed log": signed_log.skew(), "Yeo-Johnson": yeo_johnson.skew()}, name="skewness").to_frame()

zero bills: 6.7%; negative bills: 2.0%


,skewness
raw,2.664
signed log,-2.078
Yeo-Johnson,-2.448


Neither transform delivered a symmetric column. The raw skew of about +2.7 became about **−2**: the transformation *over-corrected*. The reason is visible in the first line: several percent of customers have a bill of exactly zero, which pile up in a spike that the transform cannot spread out, leaving a long tail on the *other* side. **Always look at the result of a transformation** (a histogram or the skew) rather than assuming it worked. For tree models this is moot, since they are indifferent to monotone transforms.

## 4. Categories: numbers that are not numbers

`PAY_0` is coded −2, −1, 0, 1, 2, ... for repayment status. Fed to a linear model as a number, it assumes each step has the *same* effect on the log-odds of default. But the first page showed it does not: on-time customers default about 13% to 17%, one month late 34%, two months late 69%. The fix is **one-hot encoding**: one 0/1 column per status, so each status gets its own coefficient.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

status_cols = [c for c in X.columns if c.startswith("PAY_") and not c.startswith("PAY_AMT")]            # PAY_0, PAY_2 ... PAY_6
as_numbers = ColumnTransformer([("scale", StandardScaler(), list(X.columns))])
as_categories = ColumnTransformer([("onehot", OneHotEncoder(handle_unknown="ignore"), status_cols), ("scale", StandardScaler(), [c for c in X.columns if c not in status_cols])])

print(f"logistic regression, repayment status as numbers:     AUC {auc(make_pipeline(as_numbers, LogisticRegression(max_iter=3000))):.3f}")
print(f"logistic regression, repayment status as categories:  AUC {auc(make_pipeline(as_categories, LogisticRegression(max_iter=3000))):.3f}")

logistic regression, repayment status as numbers:     AUC 0.711
logistic regression, repayment status as categories:  AUC 0.757


A clear gain from the same data and the same model, purely from representing the feature correctly. (Trees do not need it: they can split a numeric column at any threshold, so they find the 1-versus-2 jump on their own.)

Encoders also need a plan for **categories they have never seen**. `EDUCATION` has rare codes, and a category absent from the training data will appear in future data sooner or later:

In [7]:
print("EDUCATION codes and counts:", X["EDUCATION"].value_counts().sort_index().to_dict())

train_known = X_train[X_train["EDUCATION"].isin([1, 2, 3, 4])][["EDUCATION"]]
unseen = X_test[X_test["EDUCATION"].isin([5, 6])][["EDUCATION"]]                        # codes 5 and 6 never appeared in training

strict = OneHotEncoder().fit(train_known)
try:
    strict.transform(unseen)
except ValueError as e:
    print("default encoder ->", str(e)[:80])

lenient = OneHotEncoder(handle_unknown="ignore").fit(train_known)
print("handle_unknown='ignore' ->", lenient.transform(unseen).toarray()[:2].tolist(), "(all zeros: 'none of the known categories')")

grouped = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=200).fit(X[["EDUCATION"]])
print("min_frequency=200 groups rare codes together:", grouped.get_feature_names_out().tolist())

EDUCATION codes and counts: {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
default encoder -> Found unknown categories [np.int64(5), np.int64(6)] in column 0 during transform
handle_unknown='ignore' -> [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]] (all zeros: 'none of the known categories')
min_frequency=200 groups rare codes together: ['EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_5', 'EDUCATION_infrequent_sklearn']


A default encoder **raises an error** on an unseen category, which is the safe behaviour while developing but a crash in production. `handle_unknown="ignore"` encodes it as all zeros. And `min_frequency` merges rare categories into one "infrequent" bucket, which also stops very rare codes from getting unstable coefficients. Pick one deliberately.

One more practical detail: one-hot output is **sparse** by default (a memory-saving format), and some models, such as `HistGradientBoostingClassifier`, require dense input. Use `OneHotEncoder(sparse_output=False)` for those.

## 5. Missing values

Models other than a few tree implementations cannot accept `NaN`. `SimpleImputer` fills gaps with a statistic learned from the **training** data (median, mean, most frequent, or a constant). Setting `add_indicator=True` also adds a 0/1 column marking *which* values were missing, in case the missingness itself carries information.

This dataset has no gaps, so we create some: delete 20% of two columns **completely at random**, then compare strategies by cross-validation.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng = np.random.default_rng(0)
X_gappy = X.copy()
for col in ["BILL_AMT1", "PAY_AMT1"]:
    X_gappy.loc[rng.random(len(X_gappy)) < 0.2, col] = np.nan
print(f"missing: BILL_AMT1 {X_gappy['BILL_AMT1'].isna().mean():.0%}, PAY_AMT1 {X_gappy['PAY_AMT1'].isna().mean():.0%}")

cv = StratifiedKFold(5, shuffle=True, random_state=0)
def cv_auc(model, data):
    s = cross_val_score(model, data, y, cv=cv, scoring="roc_auc")
    return f"{s.mean():.4f} (+/- {s.std():.4f})"

pd.Series({
    "no gaps (the reference)": cv_auc(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), X),
    "median imputation": cv_auc(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000)), X_gappy),
    "mean imputation": cv_auc(make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(), LogisticRegression(max_iter=3000)), X_gappy),
    "median + missing indicator": cv_auc(make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), LogisticRegression(max_iter=3000)), X_gappy),
}, name="cross-validated AUC").to_frame()

missing: BILL_AMT1 20%, PAY_AMT1 20%


,cross-validated AUC
no gaps (the reference),0.7228 (+/- 0.0090)
median imputation,0.7217 (+/- 0.0099)
mean imputation,0.7218 (+/- 0.0099)
median + missing indicator,0.7218 (+/- 0.0100)


All three strategies land within a few ten-thousandths of the version with no gaps, and well inside the fold-to-fold noise. **That is the honest result:** here the gaps are random and in two weak predictors, so *how* you fill them hardly matters. The strategies differ when missingness is informative (like the Airbnb prices that were missing because a listing was unavailable, on the pandas cleaning page), which is exactly when `add_indicator=True` earns its keep.

<div class="admonition warning">
<p class="admonition-title">Fit preprocessors on the training data only</p>
<p>Every learned preprocessor (scaler, encoder, imputer) must be fitted on training rows and then applied to the rest. The safest way to guarantee that is a <code>Pipeline</code>, which is the next page.</p>
</div>

## Pitfalls at a glance

| Surprise | Cause | Fix |
|---|---|---|
| k-NN or SVM performs oddly | features on very different scales | `StandardScaler` (or `RobustScaler` with outliers) in a pipeline |
| A log transform leaves a skewed column | a spike at zero | look at the result; consider a different treatment |
| A categorical coded 0..k used as a number | assumes equal spacing and an order | one-hot encode (linear models, k-NN) |
| `ValueError: Found unknown categories` | a new category in new data | `handle_unknown="ignore"` or `min_frequency` |
| `TypeError: Sparse data was passed` | a model needs dense input | `OneHotEncoder(sparse_output=False)` |
| Imputed values leak information | statistics computed on all rows | impute inside the pipeline, fit on train |

## Check yourself

<details class="question">
<summary>1. Why does k-NN's score jump with scaling while a decision tree's does not?</summary>

k-NN decides by **distance**, so a feature measured in large units dominates the distance and drowns out the others. A tree only asks "is this feature above this threshold?", which depends on the *order* of values, not their size; any monotone rescaling leaves every possible split unchanged.

</details>

<details class="question">
<summary>2. When does <code>MinMaxScaler</code> go wrong, and what is the safer alternative?</summary>

When there are outliers: one extreme value becomes 1.0 and squeezes every ordinary value into a narrow band near 0. `RobustScaler` centres on the median and scales by the interquartile range, which barely respond to a few extreme values.

</details>

<details class="question">
<summary>3. The one-hot version of repayment status scored much higher in a logistic regression. Why would it not help a gradient-boosted model nearly as much?</summary>

A linear model gives a numeric feature a single slope, so it cannot represent "13%, 34%, then 69%". One-hot columns give each level its own coefficient. Trees split a numeric feature at any threshold, so they can isolate "2 or more months late" by themselves; the encoding adds little (gradient boosting moves by only a few thousandths).

</details>

## Exercises

<details class="question">
<summary>Compare the three scalers inside a k-NN pipeline. Which wins, and is the difference meaningful?</summary>

```python
for name, scaler in [("standard", StandardScaler()), ("minmax", MinMaxScaler()), ("robust", RobustScaler())]:
    print(name, round(auc(make_pipeline(scaler, KNeighborsClassifier(25))), 3))
```

Robust comes out slightly ahead (about 0.742, against 0.738 standard and 0.737 min-max), but a difference of a few thousandths on one split is well within noise. The real lesson is that *any* sensible scaling gets you from about 0.65 to about 0.74, and the choice among them is second-order.

</details>

<details class="question">
<summary>Use one-hot encoding for the repayment-status columns with <code>HistGradientBoostingClassifier</code>. Does it help? (Hint: the model needs dense input.)</summary>

```python
dense_onehot = ColumnTransformer([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), status_cols)], remainder="passthrough")
print(round(auc(make_pipeline(dense_onehot, HistGradientBoostingClassifier(random_state=0))), 3), "vs numeric:", round(auc(HistGradientBoostingClassifier(random_state=0)), 3))
```

It barely changes the score (about 0.777 against 0.774): the trees could already find the status thresholds. Without `sparse_output=False` you would get the "Sparse data was passed" error.

</details>

## Recap

- **Which models need scaling:** distance- and gradient-based ones (k-NN, SVM, k-means, PCA, neural nets), not trees.
- Scalers differ in how they treat **outliers**: Standard (mean/sd), MinMax (squashed by outliers), Robust (median/IQR).
- **Check** any transform's result: a spike at zero defeats a log.
- **One-hot encode** categories that are not truly ordered numbers, with a policy for unseen levels (`ignore`, `min_frequency`) and `sparse_output=False` when a model needs dense input.
- Impute with `SimpleImputer`; add an indicator when missingness may be informative.
- Every learned preprocessor is **fitted on training data only**: put them in a pipeline.

**Next:** *Pipelines*, chaining it all together safely.